Lipinski annotations on the 'Drug' Subset from the CSD
The drug subset includes FDA approved-molecules which do not necessarily comply with Lipinski's Ro5

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from upsetplot import UpSet, from_indicators
from venn import venn
from rdkit import Chem
from rdkit.Chem import Descriptors

from ccdc import io

In [3]:
#extract drug subset
drug_reader = io.EntryReader(subset=io.Subsets.DRUG)

Using csd interactor to extract the molecule object for each entry within the 'Drug' subset and the SMILES string components. 

In [ ]:
drug_output = Path("all_drug_properties.csv")
# worth making more stable
def csd_iterator(drug_reader):
    for entry in drug_reader:
        try:
            mol = entry.molecule
            components = mol.components #changed from mol.smiles
            if not components:
                continue
            
            # Only includes largest component of smiles strings to avoid analysing solvent
            # molecules and small co-crystallised components.
            largest = max(components, key=lambda c: len(c.atoms))
            smiles = largest.smiles

        except RuntimeError: 
            continue
        yield mol.identifier.strip(), smiles

Calculate the molecular properties: molecular weight, number of H-bond donors, number of H-bond acceptors, Crippen logP
Using RDKit for each entry iterated through
This will also output failed molecules into a separate json file

In [ ]:
records = []
failed_records = []

for i, (identifier, smiles) in enumerate(csd_iterator(drug_reader)):

    if not smiles:
        failed_records.append({
            "identifier": identifier,
            "stage": "missing_smiles",
            "error": "No SMILES available"
        })
        continue

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        failed_records.append({
            "identifier": identifier,
            "stage": "smiles_parse",
            "error": "RDKit could not parse SMILES"
        })
        continue

    logp_crippen = Descriptors.MolLogP(mol)
    mol_weight = Descriptors.ExactMolWt(mol)
    hb_donors = Descriptors.NumHDonors(mol)
    hb_acceptors = Descriptors.NumHAcceptors(mol)

    records.append({
        "identifier": identifier,
        "smiles": smiles,
        "HBD": hb_donors,
        "HBA": hb_acceptors,
        "logP": logp_crippen,
        "MW": mol_weight,
    })

# Generates separate records: sucessfully analysed compounds and molecules that failed during processing.
df = pd.DataFrame(records)
failed_df = pd.DataFrame(failed_records)

df.to_csv(drug_output, index=False)
failed_df.to_csv("failed_drug_properties.csv", index=False)

[15:38:47] Explicit valence for atom # 8 C, 5, is greater than permitted
[15:38:49] Explicit valence for atom # 6 C, 5, is greater than permitted
[15:38:49] Explicit valence for atom # 0 H, 2, is greater than permitted
[15:38:50] Explicit valence for atom # 35 C, 5, is greater than permitted
[15:38:51] Explicit valence for atom # 4 N, 4, is greater than permitted
[15:38:51] Explicit valence for atom # 0 O, 3, is greater than permitted
[15:38:53] Explicit valence for atom # 1 Te, 8, is greater than permitted
[15:38:55] Explicit valence for atom # 4 C, 5, is greater than permitted
[15:38:55] Explicit valence for atom # 3 B, 4, is greater than permitted
[15:38:55] Explicit valence for atom # 3 B, 4, is greater than permitted
[15:38:55] Explicit valence for atom # 3 B, 4, is greater than permitted
[15:38:55] Explicit valence for atom # 3 B, 4, is greater than permitted
[15:38:56] Explicit valence for atom # 7 N, 5, is greater than permitted
[15:38:57] Explicit valence for atom # 4 O, 3, is

In [14]:
df.head()

,identifier,smiles,HBD,HBA,logP,MW
0,ABABIQ,COc1c(N2CC3[NH2+]CCCC3C2)c(F)cc2C(=O)C(=CN(C3C...,2,5,1.34430,402.182361
1,ABADOY,[NH3+]c1ccc(cc1)S(N)(=O)=O,2,2,-0.79260,173.037925
2,ABAJAQ,O=S(=O)([O-])c1cccc2c1cccc2S(=O)(=O)[O-],0,6,0.64800,285.961677
3,ABAQEB,OC(=O)C1CCCCC1C(=O)[O-],1,3,-0.37270,171.066282
4,ABAZUB,COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23)C...,1,8,5.19038,529.164745


In [15]:
failed_df.head()

,identifier,stage,error
0,AJUKUL,smiles_parse,RDKit could not parse SMILES
1,APOQOM,smiles_parse,RDKit could not parse SMILES
2,AQOSUV,smiles_parse,RDKit could not parse SMILES
3,AVUDUS,smiles_parse,RDKit could not parse SMILES
4,BAGUBR,smiles_parse,RDKit could not parse SMILES


In [16]:
#could compare stats from separated components vs single SMILES strings for each entry
#Diana's suggestion - separate the components into refcodes. 
# I've attempted this below

# Build a second dataset with one row per component
component_records = []

for entry in drug_reader:
    try:
        mol = entry.molecule
        refcode = mol.identifier.strip()
        components = mol.components
    except RuntimeError:
        continue

    if not components:
        continue

    for component_index, component in enumerate(components, start=1):
        smiles = component.smiles
        if not smiles:
            continue

        rdkit_mol = Chem.MolFromSmiles(smiles)
        if rdkit_mol is None:
            continue

        component_records.append({
            "refcode": refcode,
            "component_refcode": f"{refcode}_{component_index}",
            "component_index": component_index,
            "smiles": smiles,
            "HBD": Descriptors.NumHDonors(rdkit_mol),
            "HBA": Descriptors.NumHAcceptors(rdkit_mol),
            "logP": Descriptors.MolLogP(rdkit_mol),
            "MW": Descriptors.ExactMolWt(rdkit_mol),
        })

component_df = pd.DataFrame(component_records)
component_df.to_csv("all_drug_properties_components.csv", index=False)

[15:53:30] Explicit valence for atom # 0 O, 4, is greater than permitted
[15:53:30] WARNING: not removing hydrogen atom without neighbors
[15:53:30] WARNING: not removing hydrogen atom without neighbors
[15:53:32] Explicit valence for atom # 0 C, 5, is greater than permitted
[15:53:32] WARNING: not removing hydrogen atom without neighbors
[15:53:32] WARNING: not removing hydrogen atom without neighbors
[15:53:32] WARNING: not removing hydrogen atom without neighbors
[15:53:32] WARNING: not removing hydrogen atom without neighbors
[15:53:33] WARNING: not removing hydrogen atom without neighbors
[15:53:33] WARNING: not removing hydrogen atom without neighbors
[15:53:33] WARNING: not removing hydrogen atom without neighbors
[15:53:33] WARNING: not removing hydrogen atom without neighbors
[15:53:33] WARNING: not removing hydrogen atom without neighbors
[15:53:33] Explicit valence for atom # 8 C, 5, is greater than permitted
[15:53:35] Explicit valence for atom # 1 C, 5, is greater than per

In [ ]:
# This creates a comparison table that compares the means for each of the Ro5 values between 
# compounds which are the whole CSD entry vs just the largest component molecular component of the entry.
comparison = pd.DataFrame({
    "Whole entry": df[["HBD", "HBA", "logP", "MW"]].mean(),
    "Components": component_df[["HBD", "HBA", "logP", "MW"]].mean()
})

comparison

,Whole entry,Components
HBD,1.724909,1.131059
HBA,3.915564,2.306251
logP,0.972018,0.040803
MW,271.619105,159.975179


Annotating the entire CSD database and compliancy with Lipinski's Ro5

In [ ]:
#Insert code here for how to calculate and extract molecular properties form the csd and output into a csv file (will not include the extracted entries on github due to IP)

In [ ]:
#find input csv file for entire csd database entries presumably containing the relevant information
csd_df = pd.read_csv("all_drug_properties.csv", index_col = 0)

In [ ]:
lipinski_columns = ["Ro5- HBD", "Ro5- HBA", "Ro5- logP", "Ro5- MW"] # this variable looks unused at this point

thresholds = {"HBD": 5, "HBA": 10, "logP": 5, "MW": 500}

for idx, row in csd_df.iterrows():
    for col, thresh in thresholds.items():
        df.loc[idx, f"{col}_bin"] = row[col] <= thresh

In [ ]:
for col in thresholds:
    df[f"{col}_bin"] = df[f"{col}_bin"].astype('int')

In [ ]:
df.head()

In [ ]:
bin_cols = ["HBD_bin", "HBA_bin", "logP_bin", "MW_bin"]
df["bin_sum"] = df[bin_cols].sum(axis=1)

In [ ]:
df

Output table to show how many of Lipinski's conditions are satisfied for each entry

In [ ]:
row_summary = (df["bin_sum"].value_counts().reindex([4,3,2,1,0], fill_value=0).to_frame(name="count"))
row_summary["percent"] = 100 * row_summary["count"] / row_summary["count"].sum()

row_summary["percent"] = row_summary["percent"].round(2)

row_summary

In [ ]:
row_summary["count"].sum() == len(df)

Number of entries satisfying each Ro5 rule

In [ ]:
col_summary = df[bin_cols].sum().to_frame(name="count")

col_summary["percent"] = 100 * col_summary["count"] / len(csd_df)

col_summary["percent"] = col_summary["percent"].round(2)

col_summary

In [ ]:
sets = {c: set(df.index[df[c] == 1]) for c in bin_cols}
plt.figure(figsize=(8, 8))
# use {size} not {count}; {percentage} is percent of the union
v = venn(sets, fmt="{size}\n({percentage:.1f}%)")
plt.title("Elliptical-style 4-set Venn")
plt.show()

Generate an UpSet plot for combination of properties that satisfy the Ro5 (not necessarily all at once)

In [ ]:
bin_cols = ["HBD_bin", "HBA_bin", "logP_bin", "MW_bin"]

#csd_df[bin_cols] = csd_df[bin_cols].fillna(0).astype(int)
df[bin_cols] = df[bin_cols].fillna(0).astype(int)

# Ensure binary / boolean
#csd_df[bin_cols] = csd_df[bin_cols].fillna(0).astype(bool)
df[bin_cols] = df[bin_cols].fillna(0).astype(bool)

# Create upset data (NO sort_by here)
#data = from_indicators(bin_cols, csd_df)
data = from_indicators(bin_cols, df)

# Plot
plt.figure(figsize=(10, 6))
up = UpSet(
    data,
    subset_size="count",
    show_counts=True,
    sort_by="cardinality"
)

up.plot()
plt.suptitle("UpSet plot for properties", y=1.02)
plt.show()